In [47]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

import pandas as pd
import itertools

Full derivation and verification of the 45.8% combined access-barrier rate is in `data_stage2.ipynb`.

In [48]:
stage1 = pd.read_csv("data/stage1_documents.csv")
stage1

,verified_by_ty,measure,subgroup,percentage,population_count,denominator,source_key,confidence_tier,notes
0,yes,lacks_easy_dpoc_access,national,9.1,21300000.0,all_voting_age_citizens,cdce_voterid_2024,published,"Report: 'Over 9% of voting-age citizens, or 21..."
1,yes,lacks_dpoc_entirely,national,1.6,3800000.0,all_voting_age_citizens,cdce_voterid_2024,derived,Report says 'just under 2%' and 'over 3.8 mill...
2,yes,lacks_easy_dpoc_access,citizens_of_color,11.0,8400000.0,all_voting_age_citizens,cdce_voterid_2024,published,over 8.4 million
3,yes,lacks_easy_dpoc_access,white_citizens,8.0,12900000.0,all_voting_age_citizens,cdce_voterid_2024,published,over 12.9 million
4,yes,lacks_dpoc_entirely,citizens_of_color,3.0,NaN,all_voting_age_citizens,cdce_voterid_2024,published,3x the white rate
5,yes,lacks_dpoc_entirely,white_citizens,1.0,NaN,all_voting_age_citizens,cdce_voterid_2024,published,NaN
6,yes,lacks_easy_dpoc_access,independents,13.0,4500000.0,all_voting_age_citizens,cdce_voterid_2024,published,almost 4.5 million
7,yes,lacks_easy_dpoc_access,democrats,10.0,9700000.0,all_voting_age_citizens,cdce_voterid_2024,published,just under 9.7 million
8,yes,lacks_easy_dpoc_access,republicans,7.0,7100000.0,all_voting_age_citizens,cdce_voterid_2024,published,"over 7.1 million. Barrier crosses party lines,..."
9,yes,lacks_easy_dpoc_access,male,11.0,12000000.0,all_voting_age_citizens,cdce_2025,published,over 12 million


In [62]:
stage2 = pd.read_csv("data/stage2_access.csv")
stage2

,measure,subgroup,percentage,population_count,denominator,source_key,confidence_tier,notes
0,reason_not_voting,too_busy_conflicting_schedule,17.8,3232700,registered_but_did_not_vote,cps_2024_table10,published,Closest proxy for 'can't physically get there'...
1,reason_not_voting,transportation_problems,2.2,399500,registered_but_did_not_vote,cps_2024_table10,published,Most direct proxy for physical access barrier
2,reason_not_voting,illness_or_disability,12.4,2252000,registered_but_did_not_vote,cps_2024_table10,published,"Physical access barrier, different mechanism t..."
3,reason_not_voting,out_of_town,7.4,1343900,registered_but_did_not_vote,cps_2024_table10,published,Access barrier - not physically present
4,reason_not_voting,registration_problems,3.6,653800,registered_but_did_not_vote,cps_2024_table10,published,"Direct hit on our registration-gate concept, t..."
5,reason_not_voting,inconvenient_polling_place,2.4,435900,registered_but_did_not_vote,cps_2024_table10,published,"Access barrier, polling-location specific"
6,reason_not_voting,not_interested,19.7,3577700,registered_but_did_not_vote,cps_2024_table10,published,Not an access barrier - included only for the ...
7,reason_not_voting,forgot_to_vote,4.1,744600,registered_but_did_not_vote,cps_2024_table10,published,Not an access barrier - included only for the ...
8,reason_not_voting,didnt_like_candidates_or_campaign,14.7,2669700,registered_but_did_not_vote,cps_2024_table10,published,Not an access barrier - included only for the ...
9,reason_not_voting,bad_weather,0.3,54500,registered_but_did_not_vote,cps_2024_table10,published,Not an access barrier - included only for the ...


# Making Funnel

In [49]:
voter_edges = [
    ("has_documents", "registered"),
    ("can_access_in_person", "registered"),
    ("registered", "voted"),
]

voter_model = DiscreteBayesianNetwork(voter_edges)

print("Nodes:", voter_model.nodes())
print("Edges:", voter_model.edges())

Nodes: ['has_documents', 'registered', 'can_access_in_person', 'voted']
Edges: [('has_documents', 'registered'), ('registered', 'voted'), ('can_access_in_person', 'registered')]


# Root Node CPT's

`has_documents` and `can_access_in_person` have no parent nodes, so each just needs a
single prior probability, taken directly from Stage 1 and Stage 2 data 

In [50]:
cpd_documents = TabularCPD(
    variable="has_documents",
    variable_card=2,
    values=[[0.909],   
            [0.091]],  
    state_names={"has_documents": ["Yes", "No"]}
)

cpd_access = TabularCPD(
    variable="can_access_in_person",
    variable_card=2,
    values=[[0.542],   
            [0.458]],  
    state_names={"can_access_in_person": ["Yes", "No"]}
)

print(cpd_documents)
print()
print(cpd_access)

+--------------------+-------+
| has_documents(Yes) | 0.909 |
+--------------------+-------+
| has_documents(No)  | 0.091 |
+--------------------+-------+

+---------------------------+-------+
| can_access_in_person(Yes) | 0.542 |
+---------------------------+-------+
| can_access_in_person(No)  | 0.458 |
+---------------------------+-------+


## Baseline turnout and registration figures

Source: U.S. Census Bureau, "Voting and Registration in the Election of November 2024"
(press release, 2025) — https://www.census.gov/newsroom/press-releases/2025/2024-presidential-election-voting-registration-tables.html

- 73.6% of voting-age U.S. citizens were registered to vote
- 65.3% of voting-age U.S. citizens voted

In [51]:
pct_registered = 0.736
pct_voted = 0.653

# CPT: registered (two parents)

`registered` depends on both `has_documents` and `can_access_in_person`
According to the Kansas *Fish v. Kobach* suspense-list case, where some registrants who had documents still got blocked due to administrative/processing failures under real world DPOC enforcement. Missing either input drops the registration probability
sharply, and missing both drops it further still.

# Kansas Fish v. Kobach — cross-checking
- Two independent sources report on the same Kansas DPOC enforcement period (2013-2016):
the ACLU reports a raw count of registrations blocked (35,000+) due to a failure to provide DPOC at the time of registration.
- About 14% of all registration attempts got blocked due to the DPOC requirement, for a mix of reasons, some of which were genuine document/access gaps and some of which were bureaucratic failure even when people did comply. We can't cleanly separate those two causes from what we currently have.
- If both are accurate, we can back-calculate the total number of registration attempts implied, as a plausibility check.

In [52]:
# ACLU's total blocked, 2013-2016
blocked_total_aclu = 35000

# court's stated ~14%
blocked_rate_court = 0.14     

implied_total_registrations = blocked_total_aclu / blocked_rate_court
successful_registrations = implied_total_registrations - blocked_total_aclu

print(f"Implied total registration attempts, 2013-2016: {implied_total_registrations:,.0f}")
print(f"Blocked: {blocked_total_aclu:,.0f}")
print(f"Implied successful registrations: {successful_registrations:,.0f}")

Implied total registration attempts, 2013-2016: 250,000
Blocked: 35,000
Implied successful registrations: 215,000


# Choosing the registered CPT values
The following table estimates the chance someone successfully registers, based on whether they have
documents and whether they can physically access a registration location in person.

- **Has documents + can access in person → 0.86 chance of registering.** 
    This is the best-case scenario, but not 1.0. We base this specifically on the Kansas *Fish v. Kobach* 
    case (2013-2016), where a documented ~14% of registration attempts were blocked even under real-world 
    enforcement, some due to genuine gaps, some due to processing errors on the government's side even when 
    documentation was submitted. See the Kansas cross-check above for the source numbers.

- **Has documents, but can't access in person → 0.10 chance of registering.** 
    The SAVE Act requires registering in person, so having the right paperwork doesn't help if someone can't
    physically get there. This number is an estimate, not pulled from a specific data source.

- **Can access in person, but doesn't have documents → 0.05 chance of registering.** 
    Missing the required documentation is treated as a slightly harder blocker than missing transportation,
    since there's no in-person workaround for not having the paperwork itself. This number is an
    estimate, not pulled from a specific data source.

- **Missing both documents and access → 0.01 chance of registering.** Essentially no path to
  registering when both requirements are unmet. This number is an estimate, not pulled from a
  specific data source.

**Limitation:** only the first column (0.86) is grounded in real historical data (Kansas,
2013-2016, one state, DMV-specific channel). The other three columns (0.10, 0.05, 0.01)
are reasoned estimates rather than measured figures, since no dataset directly measures
registration success rates split by documents-only or access-only scenarios.


In [53]:
cpd_registered = TabularCPD(
    variable="registered",
    variable_card=2,
    values=[
       [0.86,        0.10,     0.05,      0.01],
        [0.14,        0.90,     0.95,      0.99],
    ],
    evidence=["has_documents", "can_access_in_person"],
    evidence_card=[2, 2],
    state_names={
        "registered": ["Yes", "No"],
        "has_documents": ["Yes", "No"],
        "can_access_in_person": ["Yes", "No"],
    }
)

In [54]:
parent_combos = list(itertools.product(["Yes", "No"], ["Yes", "No"]))

cpt_table = pd.DataFrame(
    cpd_registered.get_values(),
    index=["registered = Yes", "registered = No"],
    columns=pd.MultiIndex.from_tuples(
        parent_combos,
        names=["has_documents", "can_access_in_person"]
    )
)

cpt_table

has_documents          Yes         No      
can_access_in_person   Yes   No   Yes    No
registered = Yes      0.86  0.1  0.05  0.01
registered = No       0.14  0.9  0.95  0.99

# CPT: voted

`voted` depends only on `registered`. Since voting requires being registered, we estimate
P(voted | registered) as the overall turnout rate divided by the overall registration rate
(65.3% / 73.6%) — this tells us, of the people who did register, what fraction went on to vote.

- **Registered → voted:** most registered people do vote, but not all. Some registered
  citizens still don't turn out for reasons unrelated to our model (forgetting,
  disliking the candidates, etc., per CPS Table 10's other reason categories).

- **Not registered → voted:** treated as 0, since voting requires registration first in
  essentially all real-world cases.


In [55]:
p_voted_given_registered = round(pct_voted / pct_registered, 3)
p_not_voted_given_registered = round(1 - p_voted_given_registered, 3)

print(f"P(voted = Yes | registered = Yes): {p_voted_given_registered}")
print(f"P(voted = No | registered = Yes): {p_not_voted_given_registered}")

P(voted = Yes | registered = Yes): 0.887
P(voted = No | registered = Yes): 0.113


In [56]:
cpd_voted = TabularCPD(
    variable="voted",
    variable_card=2,
    values=[
        [p_voted_given_registered,     0.0],   
        [p_not_voted_given_registered, 1.0],   
    ],
    evidence=["registered"],
    evidence_card=[2],
    state_names={
        "voted": ["Yes", "No"],
        "registered": ["Yes", "No"],
    }
)

print(cpd_voted)

+------------+-----------------+----------------+
| registered | registered(Yes) | registered(No) |
+------------+-----------------+----------------+
| voted(Yes) | 0.887           | 0.0            |
+------------+-----------------+----------------+
| voted(No)  | 0.113           | 1.0            |
+------------+-----------------+----------------+


# Assembling full model
validating  that everything lines up correctly (each node's CPT matches its parents, probabilities sum to 1, etc.).

In [57]:
voter_model.add_cpds(cpd_documents, cpd_access, cpd_registered, cpd_voted)

is_valid = voter_model.check_model()
print(f"Model is valid: {is_valid}")

Model is valid: True


## Inference: which stage is the bigger obstacle ?

Using Variable Elimination, we compute P(has_documents | voted = No) and
P(can_access_in_person | voted = No) given that someone didn't vote, what's the probability
they were stuck at each stage? Whichever shows a higher "No" probability indicates the bigger
obstacle.

In [58]:
inference = VariableElimination(voter_model)

# P(has_documents | voted = No)
result_documents = inference.query(
    variables=["has_documents"],
    evidence={"voted": "No"}
)
print("P(has_documents | voted = No):")
print(result_documents)
print()

# P(can_access_in_person | voted = No)
result_access = inference.query(
    variables=["can_access_in_person"],
    evidence={"voted": "No"}
)
print("P(can_access_in_person | voted = No):")
print(result_access)

P(has_documents | voted = No):
+--------------------+----------------------+
| has_documents      |   phi(has_documents) |
+====================+======================+
| has_documents(Yes) |               0.8487 |
+--------------------+----------------------+
| has_documents(No)  |               0.1513 |
+--------------------+----------------------+

P(can_access_in_person | voted = No):
+---------------------------+-----------------------------+
| can_access_in_person      |   phi(can_access_in_person) |
+===========================+=============================+
| can_access_in_person(Yes) |                      0.2805 |
+---------------------------+-----------------------------+
| can_access_in_person(No)  |                      0.7195 |
+---------------------------+-----------------------------+


# Validating

Checking whether the model's overall predicted turnout rate is reduced due to the hypithetical sinario compared to the real 2024 turnout rate (65.3%)

In [61]:
result_registered = inference.query(variables=["registered"])
print(result_registered)

+-----------------+-------------------+
| registered      |   phi(registered) |
+=================+===================+
| registered(Yes) |            0.4682 |
+-----------------+-------------------+
| registered(No)  |            0.5318 |
+-----------------+-------------------+


In [ ]:
result_voted = inference.query(variables=["voted"])
print(result_voted)

predicted_turnout = result_voted.values[0]  
actual_turnout = 0.653

difference = abs(predicted_turnout - actual_turnout)
pct_off = (difference / actual_turnout) * 100

print(f"\nModel's predicted overall turnout: {predicted_turnout:.3f}")
print(f"Real 2024 turnout rate: {actual_turnout}")
print(f"Difference: {pct_off:.1f}%")

+------------+--------------+
| voted      |   phi(voted) |
+============+==============+
| voted(Yes) |       0.4153 |
+------------+--------------+
| voted(No)  |       0.5847 |
+------------+--------------+

Model's predicted overall turnout: 0.415
Real 2024 turnout rate: 0.653
Difference: 36.4%


# Interpretation:
We can conclude that given the 88.7% rate of people who registered and voted, if the SAVE Act and executive order were in place (meaning people would have to go in person) voter registration would drop from today's 73.6% to 46.8%, causing voting turnout to drop from 65.3% to 41.5%. Based on our model, we can say that among the people who didn't vote, about 71.95% of them were more likely stuck because they couldn't come in person, rather than because they lacked documents.